# Phase VII — Full ArtBench-10 confirmatory analysis (60,000 paintings)

Final confirmatory corpus-scale run for the paper. The representations and hypotheses are frozen from Phases IV–VI; this notebook does not invent new features after seeing the complete corpus.

**Main outputs:** full B90/G44 features, full OP75 enrichment, full-corpus Phase V/Vb, artist-disjoint nested linear probes on ArtBench-10 and WikiArt-8, persistent Google Drive checkpoints, and final ZIP archives.

The longest feature-extraction step is resumable across Colab sessions.


In [ ]:
# 0. Setup: Drive + repository + dependencies
import os, sys, subprocess, shutil, tarfile, json, zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
REPO_URL='https://github.com/ardominguezm/painting-geometry.git'; BRANCH='multiscale-corpus-analysis'
REPO_DIR=Path('/content/painting-geometry')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_DIR/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ordpy>=1.2.0','kagglehub'],check=True)
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR)) if str(REPO_DIR) not in sys.path else None
import numpy as np, pandas as pd
DRIVE_ROOT=Path('/content/drive/MyDrive/painting_geometry_phase7_full'); CACHE_DIR=DRIVE_ROOT/'cache'; CHECKPOINT_DIR=DRIVE_ROOT/'checkpoints'/'B90_G44_chunks'; RESULTS_DIR=DRIVE_ROOT/'results'
DATA_DIR=Path('/content/artbench_data'); EXTRACT_DIR=DATA_DIR/'imagefolder'
for p in [DRIVE_ROOT,CACHE_DIR,CHECKPOINT_DIR,RESULTS_DIR,DATA_DIR]: p.mkdir(parents=True,exist_ok=True)
CACHE_ARTBENCH_TAR_IN_DRIVE=True; FEATURE_CHUNK_SIZE=500; ORDINAL_CHECKPOINT_EVERY=5000; N_PERMUTATIONS=4999; N_BOOTSTRAP=5000
COMMIT=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
print('Commit:',COMMIT); print('Persistent root:',DRIVE_ROOT)


## 1. Download/cache complete ArtBench-10 and official metadata

Uses the complete 256×256 ImageFolder split. The ~1.85 GB archive is cached in Google Drive by default, so later sessions can resume without another full download.


In [ ]:
import kagglehub
TAR_NAME='artbench-10-imagefolder-split.tar'; DRIVE_TAR=CACHE_DIR/TAR_NAME; LOCAL_TAR=DATA_DIR/TAR_NAME; KAGGLE_HANDLE='alexanderliao/artbench10'
train_hits=list(EXTRACT_DIR.rglob('train')) if EXTRACT_DIR.exists() else []; test_hits=list(EXTRACT_DIR.rglob('test')) if EXTRACT_DIR.exists() else []
if not (train_hits and test_hits):
    tar_path=DRIVE_TAR if DRIVE_TAR.exists() else None
    if tar_path is None:
        for candidate in [TAR_NAME,f'256X256/{TAR_NAME}',f'data/256X256/{TAR_NAME}',f'ArtBench-10/data/256X256/{TAR_NAME}']:
            try:
                p=Path(kagglehub.dataset_download(KAGGLE_HANDLE,path=candidate))
                if p.exists() and p.is_file(): tar_path=p; break
            except Exception: pass
        if tar_path is None:
            subprocess.run(['wget','-c','https://artbench.eecs.berkeley.edu/files/artbench-10-imagefolder-split.tar','-O',str(LOCAL_TAR)],check=True); tar_path=LOCAL_TAR
        if CACHE_ARTBENCH_TAR_IN_DRIVE and not DRIVE_TAR.exists(): shutil.copy2(tar_path,DRIVE_TAR)
    EXTRACT_DIR.mkdir(parents=True,exist_ok=True)
    with tarfile.open(tar_path) as tf:
        try: tf.extractall(EXTRACT_DIR,filter='data')
        except TypeError: tf.extractall(EXTRACT_DIR)
METADATA_CSV=CACHE_DIR/'ArtBench-10.csv'
if not METADATA_CSV.exists(): subprocess.run(['wget','-q','https://artbench.eecs.berkeley.edu/files/ArtBench-10.csv','-O',str(METADATA_CSV)],check=True)
assert list(EXTRACT_DIR.rglob('train')) and list(EXTRACT_DIR.rglob('test'))
print('Dataset ready:',EXTRACT_DIR); print('Metadata:',METADATA_CSV)


In [ ]:
# 2. Build and audit the 60k manifest
MANIFEST=RESULTS_DIR/'artbench_full_manifest.csv'
subprocess.run([sys.executable,'-u','scripts/prepare_artbench_manifest.py','--dataset-root',str(EXTRACT_DIR),'--metadata-csv',str(METADATA_CSV),'--output',str(MANIFEST)],check=True)
manifest=pd.read_csv(MANIFEST); print('Manifest:',manifest.shape); display(manifest.groupby(['split','style']).size().unstack(fill_value=0))
coverage=manifest['metadata_match'].mean() if 'metadata_match' in manifest else np.nan; print(f'Metadata coverage: {coverage:.2%}')
assert len(manifest)==60000,f'Expected 60000, got {len(manifest)}'


## 3. Full frozen B90 + G44 extraction with persistent chunk checkpoints

Each chunk is written to Drive. Re-running this notebook skips completed `(split, style, filename)` keys.


In [ ]:
from tqdm.auto import tqdm
from src.baselines import lbp_features,multidistance_glcm_features,multiscale_gradient_features,orientation_histogram_features
from src.curvature_v2 import relative_scale_curvature_features
from src.orientation import structure_tensor_features
from src.preprocessing import preprocess

def extract_frozen(path,long_side=256,sigma_refs=(1.,2.,4.,8.),reference_long_side=512):
    _,I=preprocess(Path(path),long_side=long_side); geom=relative_scale_curvature_features(I,long_side=long_side,sigma_refs=sigma_refs,reference_long_side=reference_long_side,return_maps=False)
    osig=2.*long_side/reference_long_side; orient=structure_tensor_features(I,sigma=osig); spx=tuple(s*long_side/reference_long_side for s in sigma_refs); base={}
    base.update(multiscale_gradient_features(I,sigmas=spx)); base.update(orientation_histogram_features(I,sigma=osig)); base.update(multidistance_glcm_features(I,distances=(1,2,4))); base.update(lbp_features(I))
    out={f'geom__curv__{k}':v for k,v in geom.items()}; out.update({f'geom__orient__{k}':v for k,v in orient.items()}); out.update({f'base__{k}':v for k,v in base.items()}); return out

def key_of(a,b,c): return f'{a}||{b}||{c}'
def done_keys():
    done=set()
    for p in sorted(CHECKPOINT_DIR.glob('chunk_*.csv')):
        try:
            d=pd.read_csv(p,usecols=['split','style','filename']); done.update(key_of(a,b,c) for a,b,c in zip(d.split,d.style,d.filename))
        except Exception as e: print('Checkpoint warning:',p,e)
    return done


In [ ]:
FULL_FEATURES=RESULTS_DIR/'artbench_full_B90_G44_features.csv'; FAILURES=RESULTS_DIR/'artbench_full_B90_G44_failures.csv'
done=done_keys(); pending=manifest[[key_of(a,b,c) not in done for a,b,c in zip(manifest.split,manifest.style,manifest.filename)]].copy(); print('Done:',len(done),'Remaining:',len(pending))
chunk_no=len(list(CHECKPOINT_DIR.glob('chunk_*.csv'))); buffer=[]; failures=[]
def flush():
    global buffer,chunk_no
    if not buffer:return
    p=CHECKPOINT_DIR/f'chunk_{chunk_no:05d}.csv'; pd.DataFrame(buffer).to_csv(p,index=False); print('CHECKPOINT',p.name,len(buffer)); buffer=[]; chunk_no+=1
for rec in tqdm(pending.itertuples(index=False),total=len(pending),desc='Full B90+G44 extraction'):
    meta=dict(split=rec.split,style=rec.style,artist=getattr(rec,'artist',''),source=getattr(rec,'source',''),filename=rec.filename,path=rec.path,long_side=256)
    try: meta.update(extract_frozen(rec.path)); buffer.append(meta)
    except Exception as e: failures.append({**meta,'error':repr(e)})
    if len(buffer)>=FEATURE_CHUNK_SIZE: flush(); pd.DataFrame(failures).to_csv(FAILURES,index=False) if failures else None
flush(); pd.DataFrame(failures).to_csv(FAILURES,index=False) if failures else None
parts=[pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('chunk_*.csv'))]; full=pd.concat(parts,ignore_index=True); full['_k']=[key_of(a,b,c) for a,b,c in zip(full.split,full.style,full.filename)]; full=full.drop_duplicates('_k').drop(columns='_k'); full.to_csv(FULL_FEATURES,index=False)
print('Full feature matrix:',full.shape); print('B/K/O=',sum(c.startswith('base__') for c in full),sum(c.startswith('geom__curv__') for c in full),sum(c.startswith('geom__orient__') for c in full))
if len(full)!=60000: print('WARNING: not yet 60k; rerun after inspecting failures.')


## 4. Full-corpus geometric organization and source sensitivity


In [ ]:
PHASE5=RESULTS_DIR/'phase7_full_style_geometry'; PHASE5B=RESULTS_DIR/'phase7_full_source_sensitivity'
if not (PHASE5/'phase5_scale_summary.csv').exists(): subprocess.run([sys.executable,'-u','scripts/run_phase5_style_geometry.py','--features',str(FULL_FEATURES),'--output-dir',str(PHASE5),'--n-permutations',str(N_PERMUTATIONS),'--seed','42'],check=True)
if not (PHASE5B/'phase5b_source_sensitivity_primary.csv').exists(): subprocess.run([sys.executable,'-u','scripts/run_phase5b_source_sensitivity.py','--features',str(FULL_FEATURES),'--output-dir',str(PHASE5B),'--n-permutations',str(N_PERMUTATIONS),'--seed','42'],check=True)
print('Phase V/Vb complete or previously cached.')


## 5. Full OP75 extraction — existing exact-validated Phase-VI implementation, resumable in Drive


In [ ]:
ENRICHED=RESULTS_DIR/'artbench_full_features_with_ordinal.csv'; ORD_CHECKPOINT=ENRICHED.with_suffix('.ordinal_checkpoint.csv')
if ORD_CHECKPOINT.exists(): print('Ordinal checkpoint rows:',len(pd.read_csv(ORD_CHECKPOINT,usecols=['__row_index'])))
if not ENRICHED.exists(): subprocess.run([sys.executable,'-u','scripts/extract_tarozo_ordinal_features.py','--features',str(FULL_FEATURES),'--dataset-root',str(EXTRACT_DIR),'--output',str(ENRICHED),'--checkpoint-every',str(ORDINAL_CHECKPOINT_EVERY)],check=True)
enriched=pd.read_csv(ENRICHED); print('Enriched:',enriched.shape,'OP75=',sum(c.startswith('ord75__') for c in enriched)); assert sum(c.startswith('ord75__') for c in enriched)==75
for c in ['ordmeta__sum75','ordmeta__sum11','ordmeta__sum24']: print(c,float(np.max(np.abs(enriched[c].to_numpy()-1))))


## 6. Confirmatory artist-disjoint nested linear probes

Linear probes are deliberate: the target is representation information, not classifier flexibility. Outer folds are artist-disjoint; `C` is selected only within training artists; pre-specified deltas use 5,000 artist-group bootstrap replicates.


In [ ]:
CONFIRM=RESULTS_DIR/'phase7_confirmatory_linear_probes'
if not (CONFIRM/'phase7_confirmatory_deltas_all.csv').exists(): subprocess.run([sys.executable,'-u',str(REPO_DIR/'scripts'/'run_phase7_full_confirmatory.py'),'--features',str(ENRICHED),'--output-dir',str(CONFIRM),'--outer-folds','5','--inner-folds','3','--n-boot',str(N_BOOTSTRAP)],check=True)
display(pd.read_csv(CONFIRM/'phase7_confirmatory_results_all.csv').round(5)); display(pd.read_csv(CONFIRM/'phase7_confirmatory_deltas_all.csv').round(5))


## 7. Build persistent archives and automatically download the compact results ZIP


In [ ]:
LIGHT=DRIVE_ROOT/'painting_geometry_phase7_full_results_LIGHT.zip'; FEATURES_ZIP=DRIVE_ROOT/'painting_geometry_phase7_feature_matrices.zip'
run_manifest={'repo_commit':COMMIT,'n_manifest':int(len(manifest)),'full_features_rows':int(len(pd.read_csv(FULL_FEATURES,usecols=['filename']))),'enriched_rows':int(len(pd.read_csv(ENRICHED,usecols=['filename']))),'n_permutations':N_PERMUTATIONS,'n_bootstrap':N_BOOTSTRAP,'feature_chunk_size':FEATURE_CHUNK_SIZE,'ordinal_checkpoint_every':ORDINAL_CHECKPOINT_EVERY}
(RESULTS_DIR/'PHASE7_RUN_MANIFEST.json').write_text(json.dumps(run_manifest,indent=2))
with zipfile.ZipFile(LIGHT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file() and p not in {FULL_FEATURES,ENRICHED,ORD_CHECKPOINT} and 'ordinal_checkpoint' not in p.name: z.write(p,p.relative_to(RESULTS_DIR))
with zipfile.ZipFile(FEATURES_ZIP,'w',zipfile.ZIP_DEFLATED) as z: z.write(FULL_FEATURES,FULL_FEATURES.name); z.write(ENRICHED,ENRICHED.name)
print('LIGHT:',LIGHT,LIGHT.stat().st_size/1e6,'MB'); print('FEATURES:',FEATURES_ZIP,FEATURES_ZIP.stat().st_size/1e6,'MB'); print('Persistent root:',DRIVE_ROOT)
files.download(str(LIGHT))


When the run finishes, send back `painting_geometry_phase7_full_results_LIGHT.zip`. That file is sufficient to revise Methods, Results, Discussion, and the main result figures. The full feature matrices remain in Drive for reproducibility.
